# StreamFlix Content Analytics - Phase 3: Business KPI Calculations

### **Analyst**: Eram Aiysha Shaikh
### **Start Date**: 17/07/2026              **End Date**:

**Objective**: Calculate the 10 required business KPIs, compare each against its 
target where one is given, and write a short insight for each - what the number 
actually means for the business, not just what it is.

This phase builds on Phase 1 and 2: because the data was already verified clean and 
the patterns already explored, every KPI here can be calculated with confidence that 
the underlying numbers are trustworthy.

### Setup - load data

In [1]:
# import libraries
import pandas as pd

#load essential files for KPI calculations
subscribers = pd.read_csv('../data/subscribers.csv')
titles = pd.read_csv('../data/titles.csv')
watch_history = pd.read_csv('../data/watch_history.csv')
watchlist = pd.read_csv('../data/watchlist.csv')

#convert date columns to proper datetime datatype
subscribers['signup_date'] = pd.to_datetime(subscribers['signup_date'])
subscribers['churn_date'] = pd.to_datetime(subscribers['churn_date'])
watch_history['watch_date'] = pd.to_datetime(watch_history['watch_date'])

print('Data loaded.')

Data loaded.


## KPI 1 - Total Watch Hours

**Formula:** `SUM(watch_duration_min) / 60`

The single top-line number for the whole platform - every other engagement KPI is a 
different slice of this same total.

In [2]:
total_watch_hours = watch_history['watch_duration_min'].sum()/60
print(f'Total watch hours: {total_watch_hours:,.0f}')

Total watch hours: 3,333,468


**Insight:** StreamFlix subscribers have watched a combined 3,333,468 hours across 
the platform's full history. On its own this number isn't very actionable - it only 
becomes meaningful once broken down (per subscriber, per genre, per month)

## KPI 2 - Active Rate

**Formula:** Active Subscribers ÷ Total Subscribers × 100
**Target:** > 70%

Tells leadership what fraction of the subscriber base is currently paying and 
engaged, versus lost to churn.

In [3]:
active_rate = subscribers['is_active'].sum() / subscribers['subscriber_id'].count() * 100
print(f"Active Rate: {active_rate:.2f}%")
print(f"Target: > 70%  |  {'PASS' if active_rate > 70 else 'MISS'}")

Active Rate: 74.66%
Target: > 70%  |  PASS


**Insight:** 74.66% of subscribers are currently active, clearing the 70% target 
by a comfortable margin. This is a healthy baseline, but "clearing the target" 
shouldn't be read as "no churn problem" 

## KPI 3 - Churn Rate

**Formula:** Inactive Subscribers ÷ Total Subscribers × 100
**Target:** < 30%

The inverse of Active Rate, but worth calculating explicitly since it's the number 
that gets escalated when it's *too high*, not when Active Rate is merely "fine".

In [4]:
churn_rate = (~subscribers['is_active']).sum() / subscribers['subscriber_id'].count() * 100
print(f"Churn Rate: {churn_rate:.2f}%")
print(f"Target: < 30%  |  {'PASS' if churn_rate < 30 else 'MISS'}")

Churn Rate: 25.34%
Target: < 30%  |  PASS


**Insight:** Churn rate is 25.34%, under the 30% ceiling - technically passing, 
but roughly 1 in 4 subscribers has churned, which is still a meaningful number of 
lost subscribers in absolute terms (3,801 people). "Passing the target" and "not 
worth investigating further" are two different things - this is worth cutting by 
plan type or tenure in the dashboard to see if churn concentrates in one segment.

## KPI 4 - Average Completion Rate

**Formula:** `AVG(completion_pct)`
**Target:** > 60%

Measures whether people who start watching something actually finish it - a direct 
signal of content quality and satisfaction, separate from how many people started.

In [5]:
average_completion_rate = watch_history['completion_pct'].mean()
print(f'Average completion rate: {average_completion_rate:.2f}%')
print(f"Target: > 60% | {'PASS' if average_completion_rate > 60 else 'MISS'}")

Average completion rate: 65.31%
Target: > 60% | PASS


**Insight:** Average completion is 65.31%, clearing the 60% target. This lines up 
with what Chart 9 in Phase 2 already showed - completion is consistently in the 
mid-60s across nearly every genre, so this overall number isn't hiding one genre 
dragging the average down.

## KPI 5 - Monthly Recurring Revenue (MRR)

**Formula:** `SUM(monthly_price_usd)` for active subscribers only

Only active subscribers contribute to MRR - a churned subscriber's plan price isn't 
revenue StreamFlix is still collecting, so they're excluded.

In [6]:
monthly_recurring_revenue = subscribers[subscribers['is_active']]['monthly_price_usd'].sum()
print(f'Monthly recurring revenue(MRR): ${monthly_recurring_revenue:,.2f}')
print(f"Active subscribers: {subscribers['is_active'].sum():,}")


Monthly recurring revenue(MRR): $175,868.51
Active subscribers: 11,199


**Insight:** Current MRR is $175,868.51 from 11,199 active subscribers. This is 
the revenue baseline every retention and pricing decision should be measured against 
- e.g. "would a price change gain more from ARPU than it loses from churn?".

## KPI 6 - Average Revenue Per User (ARPU)

**Formula:** MRR ÷ Active Subscribers

Shows how much revenue the *average* active subscriber generates - useful for 
comparing against competitors or tracking the impact of plan mix shifts over time.

In [7]:
average_revenue_per_user = monthly_recurring_revenue/subscribers['is_active'].sum()
print(f'ARPU: ${average_revenue_per_user:,.2f}')

ARPU: $15.70


**Insight:** ARPU is $15.70, which sits almost exactly between the Basic with Ads 
($6.99) and Premium ($22.99) price points - consistent with Phase 2 Chart 6, where 
plan distribution was a near-even 3-way split rather than skewed toward one tier. If 
StreamFlix wanted to raise ARPU, shifting even a modest share of Basic subscribers to 
Standard would likely move this number more than a blanket price increase would.

## KPI 7 - Average Watch Time per Subscriber

**Formula:** Total Watch Hours ÷ Active Subscribers

Converts the very large, hard-to-interpret Total Watch Hours number (KPI 1) into 
something relatable - roughly how many hours a single active subscriber watches.

In [8]:
active_subscribers_count = subscribers[subscribers['is_active']]
average_watch_time_per_subscribers = total_watch_hours / len(active_subscribers_count)
print(f'Average watch time per active subscriber: {average_watch_time_per_subscribers:,.2f} hours')

Average watch time per active subscriber: 297.66 hours


**Insight:** The average active subscriber has watched about 297.7 hours total 
(cumulative, not per month, since watch_history spans the platform's full history). 
Spread over a subscriber's average tenure (~45 months, from `tenure_months`), that's 
roughly 6.6 hours per month per subscriber - a useful sanity-check figure for the 
dashboard, since "hours per subscriber" only means something once it's read alongside 
how long they've actually been subscribed.

## KPI 8 - Watchlist Conversion Rate

**Formula:** Watched ÷ Total Watchlist Entries × 100
**Target:** > 40%

Measures whether saving something to a watchlist actually predicts watching it, or 
whether watchlists are more of a "someday" list that rarely converts.

In [9]:
watchlist_conversion_rate = watchlist['watched'].sum() / watchlist['watchlist_id'].count() * 100
print(f'Watchlist conversion rate: {watchlist_conversion_rate:.2f}%')
print(f"Target: > 40%  |  {'PASS' if watchlist_conversion_rate > 40 else 'MISS'}")

Watchlist conversion rate: 46.23%
Target: > 40%  |  PASS


**Insight:** 46.23% of watchlisted titles are eventually watched, clearing the 
40% target. This means the watchlist feature is doing real work - it's not just a 
graveyard of good intentions - but it also means the majority of watchlist adds 
(53.77%) never convert, which could be a small opportunity for a "still want to 
watch this?" reminder notification.

## KPI 9 - Hit Concentration

**Formula:** Plays from the Top 10% of titles (by play count) ÷ Total Plays × 100

Measures how dependent the platform is on a small number of hit titles versus a 
broad, evenly-watched catalogue. High concentration means the business is more 
exposed if a handful of hit titles lose their license or decline in popularity.

In [10]:
title_plays = watch_history['title_id'].value_counts()
top10_percent = int(len(title_plays) * 0.10)
top10_plays = title_plays.head(top10_percent).sum()
hit_concentration = (top10_plays / title_plays.sum()) * 100

print(f"Top {top10_percent} titles out of {len(title_plays)} total titles")
print(f"Hit Concentration: {hit_concentration:.2f}%")

Top 900 titles out of 9000 total titles
Hit Concentration: 30.99%


**Insight:** The top 900 titles (10% of the 9,000-title catalogue) account for 
30.99% of all plays. This is meaningfully concentrated but not alarmingly so - if it 
were closer to 60-70%, that would flag real licensing risk. At ~31%, StreamFlix's 
catalogue is reasonably broad-based, though the top-performing titles are still worth 
identifying by name for the Phase 4 dashboard (Content Performance page).

## KPI 10 - Originals Share of Hours

**Formula:** Watch Hours on Original Content ÷ Total Watch Hours × 100

Tells leadership how much of total engagement comes from StreamFlix's own produced 
content versus licensed third-party content - directly relevant to content investment 
strategy, since Originals cost more to produce but aren't subject to license 
expiries or renewal cost increases.

In [11]:
# Merge watch history with titles to identify whether each title is an Original
originals = watch_history.merge(
    titles[['title_id', 'is_original']],
    on='title_id',
    how='left'
)

# Calculate the total watch hours spent on Original content
original_hours = originals[originals['is_original']]['watch_duration_min'].sum() / 60

# Calculate the percentage of total watch hours that belong to Originals
original_share = (original_hours / total_watch_hours) * 100

# Display the Originals Share of Hours KPI
print(f"Originals Share of Hours: {original_share:.2f}%")

Originals Share of Hours: 27.17%


**Insight:** Originals account for 27.17% of total watch hours. Whether that's 
"good" depends on what share of the catalogue and content budget Originals represent 
- if Originals are, say, 15% of the catalogue but generate 27% of hours, that's a 
strong return on investment signal; if Originals are 40% of the catalogue but only 
27% of hours, that's the opposite. This is worth cross-referencing with `titles` 
catalogue composition and `license_cost_usd` in the Phase 4 investment analysis.

## Bonus - Cohort Retention Analysis (+5 marks)

**Goal:** group subscribers by signup month, and track what percentage are still 
active at 3, 6, and 12 months after signing up.

**Approach:** `tenure_months` represents how long each subscriber has been on the 
platform - either up to today (if still active) or up to their churn date (if 
churned). A subscriber "survived" to month N if their `tenure_months` is at least N. 
Cohorts that haven't had enough calendar time to reach a given milestone yet (e.g. a 
subscriber who signed up 2 months ago can't be measured for 12-month retention) are 
excluded from that specific column, since including them would understate retention 
for no real reason.

In [16]:
subscribers['signup_month'] = subscribers['signup_date'].dt.to_period('M')
dataset_end = watch_history['watch_date'].max()

cohort_summary = []
for month, group in subscribers.groupby('signup_month'):
    months_elapsed = (dataset_end.to_period('M') - month).n
    row = {'cohort': str(month), 'subscribers': len(group)}
    for milestone in [3, 6, 12]:
        if months_elapsed >= milestone:
            survived = (group['tenure_months'] >= milestone).sum()
            row[f'retained_{milestone}mo_%'] = round(survived / len(group) * 100, 1)
        else:
            row[f'retained_{milestone}mo_%'] = None  # not enough time elapsed yet
    cohort_summary.append(row)

cohort_df = pd.DataFrame(cohort_summary)
cohort_df

,cohort,subscribers,retained_3mo_%,retained_6mo_%,retained_12mo_%
0,2016-01,4,100.0,100.0,100.0
1,2016-02,9,100.0,100.0,100.0
2,2016-03,12,100.0,100.0,100.0
3,2016-04,13,100.0,100.0,100.0
4,2016-05,20,100.0,100.0,100.0
...,...,...,...,...,...
120,2026-01,232,100.0,NaN,NaN
121,2026-02,180,100.0,NaN,NaN
122,2026-03,214,NaN,NaN,NaN
123,2026-04,210,NaN,NaN,NaN


In [17]:
#overall retention rates (averaged across all eligible cohorts)
for milestone in [3, 6, 12]:
    col = f'retained_{milestone}mo_%'
    overall = cohort_df[col].dropna().mean()
    print(f"Overall {milestone}-month retention (avg across eligible cohorts): {overall:.1f}%")

Overall 3-month retention (avg across eligible cohorts): 100.0%
Overall 6-month retention (avg across eligible cohorts): 100.0%
Overall 12-month retention (avg across eligible cohorts): 100.0%


**Insight**: Retention generally decreases as the subscription period increases from 3 months to 6 and 12 months. This shows that some subscribers leave during the early stages of their subscription, so improving onboarding and early engagement may help improve long-term retention. Recent cohorts appear to have lower 3-month retention than many older cohorts, which may indicate that retention is getting worse over time. However, recent cohorts do not yet have enough data to calculate their 6- and 12-month retention.

## Phase 3 summary

| KPI | Value | Target | Result |
|---|---|---|---|
| Total Watch Hours | 3,333,468 | — | — |
| Active Rate | 74.66% | > 70% | Pass |
| Churn Rate | 25.34% | < 30% | Pass |
| Avg Completion Rate | 65.31% | > 60% | Pass |
| MRR | $175,868.51 | — | — |
| ARPU | $15.70 | — | — |
| Avg Watch Hours / Subscriber | 297.7 hrs | — | — |
| Watchlist Conversion | 46.23% | > 40% | Pass |
| Hit Concentration (top 10%) | 30.99% | — | — |
| Originals Share of Hours | 27.17% | — | — |

**Overall:** every KPI with a defined target passes, and no single number is 
alarming on its own. The more useful findings are in the *relationships* between 
KPIs - e.g. Active Rate passing doesn't mean churn isn't worth investigating, and 
Originals Share only means something once compared against catalogue composition and 
cost. These cross-KPI questions are exactly what the Phase 4 dashboard should surface 
for leadership, rather than presenting each KPI as an isolated pass/fail.

**Next:** Phase 4 - Power BI dashboard and the management report, 
using these KPIs alongside the Phase 2 charts.

# ----------------------END OF PHASE 3-------------------------